# DataMate — fine-tune trên train chính thức, rồi trả lời public_test

Luồng chạy một lần (Save & Run All). Chưa huấn luyện qua bản sửa này; cần chạy trên Kaggle 2×T4:
1. Model nền (AWQ) chạy hết **3000 câu train** trên 2 GPU (~2 giờ) → giữ các câu đáp án khớp.
2. Tạo dữ liệu SFT từ các câu đúng; **giữ lại 6 business** không dùng để huấn luyện.
3. QLoRA trên 2×T4 ((thời gian phụ thuộc dữ liệu/GPU)) → adapter LoRA.
4. So điểm +1/−1 của model nền và model đã tinh chỉnh trên 6 business giữ lại; **tự chọn bản tốt hơn**.
5. Chạy 1000 câu public_test bằng bản đã chọn → `/kaggle/working/submission.jsonl`.

Cài đặt: Accelerator **GPU T4 x2**, Internet **On** (cần cho cell tải trọng số fp16 và cài thư viện), Persistence: Files only.
Input: Dataset dữ liệu chính thức (thư mục `train/`, `public_test/`) và, nếu có, Dataset trọng số AWQ.
Tổng thời gian ước tính 5–6 giờ (giới hạn phiên 12 giờ). Đây là luồng chưa từng chạy: nếu một cell dừng với `assert`, đọc thông báo.

In [ ]:
# Cell 1 — lấy source code MỚI NHẤT (clone lần đầu; nếu thư mục đã có thì fetch + reset về origin/BRANCH)
import os, subprocess, pathlib
REPO_URL = "https://github.com/HoL0ngg/Zalo-AI-Challenge-2026.git"
BRANCH = "main"
ROOT = pathlib.Path("/kaggle/working/datamate")

def git(*args, cwd=None):
    return subprocess.run(["git", *args], cwd=cwd, capture_output=True, text=True, check=True).stdout.strip()

if (ROOT / ".git").exists():
    git("remote", "set-url", "origin", REPO_URL, cwd=ROOT)
    git("fetch", "--depth", "1", "origin", BRANCH, cwd=ROOT)
    git("reset", "--hard", "FETCH_HEAD", cwd=ROOT)      # bỏ thay đổi cục bộ, lấy đúng code trên GitHub
    git("clean", "-fd", "-e", "models/", cwd=ROOT)       # giữ symlink trọng số ở models/
else:
    if ROOT.exists():
        import shutil; shutil.rmtree(ROOT)               # thư mục dở dang, không phải repo
    git("clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(ROOT))
os.chdir(ROOT)
remote = git("ls-remote", REPO_URL, BRANCH).split()[0]   # commit thật sự đang ở đầu nhánh trên GitHub
local = git("rev-parse", "HEAD", cwd=ROOT)
print("local :", git("log", "--oneline", "-1", cwd=ROOT))
print("remote:", remote[:7], "(đầu nhánh", BRANCH, "trên GitHub)")
assert local == remote, "Code chưa mới nhất: kiểm tra đã git push chưa và BRANCH đúng chưa"
# P0 (sql_parser) và data parallel (dataparallel) phải có, nếu không là đang chạy code cũ.
for needed in ("src/datamate/codegen/sql_parser.py", "src/datamate/runtime/dataparallel.py"):
    assert (ROOT / needed).exists(), f"Thiếu {needed}: chưa push code lên GitHub"
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Cell 1b — tìm bộ dữ liệu chính thức (train có đáp án, public_test không có đáp án)
# Cần Kaggle Dataset bạn đã upload: mỗi gói là một thư mục tên `train` / `public_test`, chứa questions.jsonl + rulebook.md + calendar.csv + data/.
import pathlib

def find_packages(root="/kaggle/input"):
    found = {}
    for q in pathlib.Path(root).rglob("questions.jsonl"):
        d = q.parent
        if (d / "rulebook.md").exists() and (d / "data").is_dir():
            found[d.name] = d
    return found

pk = find_packages()
# Không có fallback sang dataset/ của repo: repo chỉ chứa dữ liệu tổng hợp cũ, chạy nhầm sẽ cho số vô nghĩa.
assert "train" in pk and "public_test" in pk, (
    "Không thấy Kaggle Dataset chính thức. Add Dataset (chứa thư mục train/ và public_test/) làm Input rồi chạy lại. "
    f"Tìm thấy: {sorted(pk)}")
TRAIN_DIR, TEST_DIR = pk["train"], pk["public_test"]
TRAIN_DIR, TEST_DIR = pathlib.Path(TRAIN_DIR), pathlib.Path(TEST_DIR)
for name, d in (("train", TRAIN_DIR), ("public_test", TEST_DIR)):
    print(name, "->", d, "| questions:", sum(1 for _ in open(d / "questions.jsonl", encoding="utf-8")) if (d / "questions.jsonl").exists() else "MISSING")
assert (TRAIN_DIR / "answers.jsonl").exists(), "train phải có answers.jsonl"
assert not (TEST_DIR / "answers.jsonl").exists(), "public_test không được có đáp án"


In [ ]:
# Cell 2 — trọng số mô hình.
# Lần đầu: tải từ Hugging Face (Internet On), rồi Save Version -> Output -> "New Dataset"
# để các lần sau add làm Input và chạy offline.
# Tự tìm thư mục chứa config.json + *.safetensors dưới /kaggle/input (Kaggle có thể đặt
# dataset ở /kaggle/input/<slug>/ hoặc /kaggle/input/datasets/<user>/<slug>/, có hoặc không có thư mục con).
import pathlib
MODEL_ID = "Qwen/Qwen2.5-Coder-7B-Instruct-AWQ"

def find_weights(root="/kaggle/input"):
    hits = []
    for cfg in pathlib.Path(root).rglob("config.json"):
        folder = cfg.parent
        if any(folder.glob("*.safetensors")) and (folder / "tokenizer.json").exists():
            hits.append(folder)
    return sorted(hits, key=lambda d: len(str(d)))

found = find_weights()
if found:
    weights = found[0]
    print("Dùng trọng số có sẵn:", weights)
    if len(found) > 1:
        print("Các thư mục khác cũng khớp:", found[1:])
else:
    weights = pathlib.Path("/kaggle/working/Qwen2.5-Coder-7B-Instruct-AWQ")
    if not (weights / "config.json").exists():
        from huggingface_hub import snapshot_download
        snapshot_download(MODEL_ID, local_dir=str(weights))
    print("Đã tải trọng số về:", weights, "— hãy Save Version và tạo Dataset từ Output")

link = ROOT / "models" / "qwen-coder-awq"          # phải khớp local_path trong configs/models_kaggle.yaml
link.parent.mkdir(exist_ok=True)
if link.is_symlink() or link.exists():
    link.unlink()
link.symlink_to(weights, target_is_directory=True)
assert (link / "config.json").exists(), f"Symlink trọng số chưa đúng: {link} -> {weights}"
print("models/qwen-coder-awq ->", link.resolve())
print(sorted(f.name for f in weights.iterdir())[:20])

In [ ]:
# Cell 3 — cài phụ thuộc (5–10 phút). vLLM kéo theo bản torch của riêng nó (hiện là cu130).
%pip install -q -e ".[test]"
%pip install -q rapidfuzz Unidecode sqlglot
%pip install -q vllm
# Kaggle cài sẵn torchaudio/torchvision build cho CUDA cũ hơn; lệch CUDA với torch mới sẽ làm
# transformers crash khi import. Dự án không dùng audio nên gỡ torchaudio; torchvision chỉ giữ
# nếu đúng bản CUDA của torch.
%pip uninstall -q -y torchaudio
import sys
import torch
# torchvision phải import được VÀ khớp bản torch: vLLM >= 0.32 import torchvision ở bước warmup kernel
# (thiếu là "No module named 'torchvision'" khi khởi động engine). Không so chuỗi version CUDA:
# torchvision.version.cuda = "13000" còn torch.version.cuda = "13.0" nên luôn lệch.
def torchvision_ok():
    import importlib
    try:
        for name in [m for m in sys.modules if m == "torchvision" or m.startswith("torchvision.")]:
            del sys.modules[name]
        importlib.import_module("torchvision.ops")   # nạp extension C++, lỗi nếu lệch torch/CUDA
        from torchvision.transforms import InterpolationMode  # noqa: F401
        return True
    except Exception as exc:  # noqa: BLE001
        print("torchvision không dùng được:", repr(exc)[:200])
        return False

if not torchvision_ok():
    minor = int(torch.__version__.split(".")[1])          # torch 2.N <-> torchvision 0.(N+15)
    tag = "cu" + torch.version.cuda.replace(".", "")
    !{sys.executable} -m pip uninstall -q -y torchvision
    !{sys.executable} -m pip install -q --no-deps "torchvision==0.{minor + 15}.*" --index-url https://download.pytorch.org/whl/{tag}
assert torchvision_ok(), "torchvision vẫn lỗi: vLLM sẽ không khởi động được engine"
import torchvision
print("torchvision", torchvision.__version__)
import vllm
print("vllm", vllm.__version__, "| torch", torch.__version__, "| cuda", torch.cuda.is_available())
import transformers.audio_utils  # phải import được mà không lỗi
print("transformers", transformers.__version__, "OK")
%pip install -q peft bitsandbytes accelerate
import peft, bitsandbytes
print('peft', peft.__version__, '| bitsandbytes', bitsandbytes.__version__)

In [ ]:
# Cell 4 — trọng số fp16 của model nền để huấn luyện (QLoRA cần bản chưa lượng tử). ~15 GB, để ở /tmp (working chỉ có 20 GB)
import os, shutil, pathlib
BASE_FP16 = pathlib.Path("/tmp/qwen-fp16")
hits = [c.parent for c in pathlib.Path("/kaggle/input").rglob("config.json")
        if "quantization_config" not in c.read_text() and any(c.parent.glob("*.safetensors")) and "coder" in str(c.parent).lower()]
if hits:
    BASE_FP16 = hits[0]
elif not (BASE_FP16 / "config.json").exists():
    free = shutil.disk_usage("/tmp").free / 1e9
    assert free > 25, f"/tmp chỉ còn {free:.0f} GB, cần > 25 GB"
    from huggingface_hub import snapshot_download
    snapshot_download("Qwen/Qwen2.5-Coder-7B-Instruct", local_dir=str(BASE_FP16), allow_patterns=["*.json", "*.safetensors", "*.txt"])
print("model nền fp16:", BASE_FP16, "| safetensors:", len(list(BASE_FP16.glob("*.safetensors"))))

In [ ]:
# Cell 5 — GIAI ĐOẠN 1: model nền chạy hết 3000 câu train (2 GPU). Mỗi câu: 1 SQL + tối đa 2 vòng sửa lỗi.
import os, time, json, pathlib
os.environ.update(HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1", VLLM_NO_USAGE_STATS="1", PYTHONHASHSEED="20261030")
R1 = "/kaggle/working/result/stage1"
_t0 = time.time()
!python -m datamate.runtime.dataparallel --config configs/kaggle.yaml     --data-dir {TRAIN_DIR} --result-dir {R1} --gpus 0,1
print(f"giai đoạn 1: {(time.time() - _t0) / 60:.0f} phút")
_bad = [p.name for p in sorted(pathlib.Path(R1, ".shards").glob("shard_*.log")) if "Traceback" in p.read_text(errors="ignore")]
assert not _bad, f"Shard lỗi: {_bad} — đọc log trong {R1}/.shards/"
!python -m datamate.cli eval {R1}/submission.jsonl {TRAIN_DIR}/answers.jsonl > /kaggle/working/eval_stage1.json
!python -m datamate.cli report {R1} {TRAIN_DIR}/answers.jsonl > /kaggle/working/report_stage1.json
!head -c 1800 /kaggle/working/report_stage1.json

In [ ]:
# Cell 6 — GIAI ĐOẠN 2: dữ liệu SFT từ các câu model nền trả lời ĐÚNG (rejection sampling); giữ lại 6 business
!python scripts/build_sft_data.py {R1} {TRAIN_DIR} --out /kaggle/working/sft/train.jsonl --holdout-out /kaggle/working/sft/holdout.json
import json
HOLDOUT = json.load(open("/kaggle/working/sft/holdout.json"))["businesses"]
N_SFT = sum(1 for _ in open("/kaggle/working/sft/train.jsonl", encoding="utf-8"))
print("ví dụ huấn luyện:", N_SFT, "| business giữ lại:", HOLDOUT)
assert N_SFT >= 300, f"Chỉ có {N_SFT} ví dụ đúng: model nền quá yếu để học từ chính nó, xem report_stage1.json"

In [ ]:
# Cell 7 — QLoRA training; fail fast and retain full log
import os, subprocess, pathlib, time
from pathlib import Path
assert Path("/kaggle/working/sft/train.jsonl").is_file(), "SFT data missing"
assert (Path(BASE_FP16) / "config.json").is_file(), "FP16 base weights missing"
cmd = ["torchrun", "--standalone", "--nproc_per_node=2",
       "scripts/train_lora.py", "--base", str(BASE_FP16),
       "--data", "/kaggle/working/sft/train.jsonl", "--out", "/kaggle/working/lora"]
log = Path("/kaggle/working/qlora_train.log")
start = time.monotonic()
with log.open("w", encoding="utf-8") as f:
    run = subprocess.run(cmd, cwd=ROOT, stdout=f, stderr=subprocess.STDOUT,
                         env=dict(os.environ, TOKENIZERS_PARALLELISM="false"))
print("LoRA exit code:", run.returncode, "minutes:", round((time.monotonic()-start)/60,1))
print(log.read_text(encoding="utf-8", errors="replace")[-12000:])
if run.returncode:
    raise RuntimeError(f"torchrun failed ({run.returncode}); see {log}")
adapter = Path("/kaggle/working/lora")
for file in ("adapter_config.json", "adapter_model.safetensors"):
    assert (adapter / file).is_file(), f"Missing {file}; see {log}"
link = ROOT / "models" / "lora"
link.parent.mkdir(parents=True, exist_ok=True)
if link.is_symlink() or link.is_file(): link.unlink()
elif link.exists(): raise RuntimeError(f"Refusing to overwrite directory: {link}")
link.symlink_to(adapter, target_is_directory=True)
print("LoRA adapter:", link.resolve())


In [ ]:
# Cell 8 — GIAI ĐOẠN 4: so model nền và model đã tinh chỉnh trên 6 business KHÔNG dùng để huấn luyện (30 câu mỗi business)
import json
R2 = "/kaggle/working/result/holdout_lora"
import subprocess, sys
cmd = [sys.executable, "-m", "datamate.runtime.dataparallel",
       "--config", "configs/kaggle_lora.yaml",
       "--data-dir", str(TRAIN_DIR), "--result-dir", R2,
       "--gpus", "0,1", "--max-per-business", "30", "--business", *HOLDOUT]
proc = subprocess.run(cmd, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print(proc.stdout[-10000:])
if proc.returncode:
    for p in sorted(pathlib.Path(R2, ".shards").glob("shard_*.log")):
        print(p.name, p.read_text(errors="replace")[-9000:])
    raise RuntimeError(f"LoRA holdout failed: {proc.returncode}")
def rows(p): return [json.loads(l) for l in open(p, encoding="utf-8")]
tuned = {r["id"]: r for r in rows(f"{R2}/submission.jsonl")}
base = {r["id"]: r for r in rows(f"{R1}/submission.jsonl")}
gold = {r["id"]: r for r in rows(f"{TRAIN_DIR}/answers.jsonl")}
from datamate.eval.scorer import matches
def points(pred, ids):
    right = sum(1 for i in ids if i in pred and pred[i]["type"] != "abstain" and matches(pred[i], gold[i]))
    wrong = sum(1 for i in ids if i in pred and pred[i]["type"] != "abstain" and not matches(pred[i], gold[i]))
    return right, wrong, right - wrong
ids = [i for i in tuned]
print("câu so sánh:", len(ids))
base_pts, tuned_pts = points(base, ids), points(tuned, ids)
print("model nền       đúng/sai/điểm:", base_pts)
print("model tinh chỉnh đúng/sai/điểm:", tuned_pts)
USE_TUNED = tuned_pts[2] > base_pts[2]
CONFIG = "configs/kaggle_lora.yaml" if USE_TUNED else "configs/kaggle.yaml"
print("=> dùng:", "MODEL TINH CHỈNH" if USE_TUNED else "MODEL NỀN (tinh chỉnh không tốt hơn)", CONFIG)

In [ ]:
# Cell 9 — GIAI ĐOẠN 5: public_test (1000 câu, không có đáp án) bằng bản đã chọn -> /kaggle/working/submission.jsonl
import os, time, shutil, json, pathlib
from collections import Counter
RT = "/kaggle/working/result/public_test"
_t0 = time.time()
!python -m datamate.runtime.dataparallel --config {CONFIG}     --data-dir {TEST_DIR} --result-dir {RT} --gpus 0,1
print(f"public_test: {(time.time() - _t0) / 60:.0f} phút")
_bad = [p.name for p in sorted(pathlib.Path(RT, ".shards").glob("shard_*.log")) if "Traceback" in p.read_text(errors="ignore")]
assert not _bad, f"Shard lỗi: {_bad} — đọc log trong {RT}/.shards/"
!python -m datamate.cli validate {RT}/submission.jsonl
ids = {json.loads(l)["id"] for l in open(f"{TEST_DIR}/questions.jsonl", encoding="utf-8")}
sub = [json.loads(l) for l in open(f"{RT}/submission.jsonl", encoding="utf-8")]
print("câu trả lời:", len(sub), "/", len(ids), "| thiếu (tính abstain):", len(ids - {r["id"] for r in sub}), "|", dict(Counter(r["type"] for r in sub)))
shutil.copy(f"{RT}/submission.jsonl", "/kaggle/working/submission.jsonl")

In [ ]:
# Cell 10 — đóng gói để tải về (Output -> datamate_results.zip): báo cáo, adapter, log; KHÔNG kèm dữ liệu huấn luyện đầy đủ
!cd /kaggle/working && zip -rq datamate_results.zip report_*.json eval_*.json submission.jsonl lora/adapter_config.json lora/adapter_model.safetensors sft/holdout.json result -x "result/*/.shards/in_*"
!ls -la /kaggle/working/datamate_results.zip